In [1]:
from bs4 import BeautifulSoup
import requests

In [2]:
url = "https://myanimelist.net/topmanga.php"
page=requests.get(url)

In [4]:
page

<Response [200]>

In [5]:
page.text

'\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"\n    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n\n<html lang="en" class="appearance-none">\n<head>\n    \n<link rel="preconnect" href="//www.googletagmanager.com/" crossorigin="anonymous"/>\n<link rel="preconnect" href="https://cdn.myanimelist.net" crossorigin="anonymous"/>\n<title>\n  Top Manga - MyAnimeList.net\n</title>\n<meta name="description" content="Browse the highest-ranked manga on MyAnimeList, the internet&#039;s largest manga database. Find the top manga, novels, one-shots and more!" />\n\n  \n<meta name="keywords" content="anime, myanimelist, anime news, manga" />\n  \n\n        \n          \n          \n  <link rel="next" href="https://myanimelist.net/topmanga.php?limit=50" />\n\n\n\n\n<meta property="og:locale" content="en_US"><meta property="fb:app_id" content="360769957454434"><meta property="og:site_name" content="MyAnimeList.net"><meta name="twitter:card" content="summary"><meta name="

In [9]:
soup=BeautifulSoup(page.text)

In [11]:
soup.find_all("h3",class_="manga_h3")[0].text

'Berserk'

In [13]:
name=[]
for i in soup.find_all("h3",class_="manga_h3"):
    name.append(i.text)

In [15]:
details=[]
for i in soup.find_all("div",class_="information di-ib mt4"):
    details.append(i.text)

In [17]:
rating=[]
for j in [9,8,7,6,5,4,3,2,1,0]:
    for i in soup.find_all("span",class_=f"text on score-label score-{j}"):
        rating.append(i.text)

In [19]:
rank=[]
for j in range(0,5):
    for i in soup.find_all("span",class_=f"lightLink top-anime-rank-text rank{j}"):
        rank.append(i.text)

In [21]:
for i in [50,100,150,200,250,300,350,400,450,500,550,600]:
    url1 = f"https://myanimelist.net/topmanga.php?limit={i}"
    page=requests.get(url1)
    soup=BeautifulSoup(page.text)
    for i in soup.find_all("h3",class_="manga_h3"):
        name.append(i.text)
    for i in soup.find_all("div",class_="information di-ib mt4"):
        details.append(i.text)
    for j in [9,8,7,6,5,4,3,2,1,0]:
        for i in soup.find_all("span",class_=f"text on score-label score-{j}"):
            rating.append(i.text)
    for j in range(0,5):
        for i in soup.find_all("span",class_=f"lightLink top-anime-rank-text rank{j}"):
            rank.append(i.text)

In [45]:
import pandas as pd
df=pd.DataFrame({"rank":rank,
                 'name':name,
                "details":details,
                "rating":rating})
df

,rank,name,details,rating
0,1,Berserk,\n Manga (? vols)\n Aug 1989 - \...,9.47
1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,\n Manga (24 vols)\n Jan 2004 - ...,9.32
2,3,Vagabond,\n Manga (37 vols)\n Sep 1998 - ...,9.26
3,4,One Piece,\n Manga (? vols)\n Jul 1997 - \...,9.22
4,5,Monster,\n Manga (18 vols)\n Dec 1994 - ...,9.16
...,...,...,...,...
645,646,Goukon ni Ittara Onna ga Inakatta Hanashi,\n Manga (? vols)\n Feb 2021 - \...,8.07
646,647,Hitoribocchi no ○○ Seikatsu,\n Manga (8 vols)\n Sep 2013 - A...,8.07
647,648,Nogi Wakaba wa Yuusha de Aru,\n Light Novel (2 vols)\n Jul 20...,8.07
648,649,Orange,\n Light Novel (3 vols)\n Jul 20...,8.07


In [25]:
top_5_rated = df[['name', 'rating']].sort_values(by='rating', ascending=False).head(35)
top_5_rated

,name,rating
0,Berserk,9.47
1,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32
2,Vagabond,9.26
3,One Piece,9.22
4,Monster,9.16
5,Slam Dunk,9.08
6,Vinland Saga,9.08
7,Fullmetal Alchemist,9.04
10,Grand Blue,9.03
8,Tian Guan Cifu,9.03


In [84]:
df.to_csv(r'C:\Users\dtrri\bot\temp.csv')

In [194]:
len(df)

650

In [47]:
df['details']

0      \n        Manga (? vols)\n        Aug 1989 - \...
1      \n        Manga (24 vols)\n        Jan 2004 - ...
2      \n        Manga (37 vols)\n        Sep 1998 - ...
3      \n        Manga (? vols)\n        Jul 1997 - \...
4      \n        Manga (18 vols)\n        Dec 1994 - ...
                             ...                        
645    \n        Manga (? vols)\n        Feb 2021 - \...
646    \n        Manga (8 vols)\n        Sep 2013 - A...
647    \n        Light Novel (2 vols)\n        Jul 20...
648    \n        Light Novel (3 vols)\n        Jul 20...
649    \n        Manga (3 vols)\n        Feb 2011 - M...
Name: details, Length: 650, dtype: object

In [51]:
# Function to extract details from the "details" column
def extract_details(detail_text):
    parts = detail_text.strip().split("\n")  # Split by new lines
    format_type = parts[0].strip() if len(parts) > 0 else None
    volume_info = parts[1].strip() if len(parts) > 1 else None
    publication_dates = parts[2].strip() if len(parts) > 2 else None
    
    return pd.Series([format_type, volume_info, publication_dates])

# Apply extraction function to details column
df[['Format', 'Volume Info', 'Publication Dates']] = df['details'].apply(extract_details)

# Drop the original 'details' column
df_cleaned = df.drop(columns=['details'])

# Display cleaned dataset
df_cleaned.head()


,rank,name,rating,Format,Volume Info,Publication Dates
0,1,Berserk,9.47,Manga (? vols),Aug 1989 -,"741,230 members"
1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga (24 vols),Jan 2004 - Apr 2011,"287,722 members"
2,3,Vagabond,9.26,Manga (37 vols),Sep 1998 - May 2015,"417,353 members"
3,4,One Piece,9.22,Manga (? vols),Jul 1997 -,"654,245 members"
4,5,Monster,9.16,Manga (18 vols),Dec 1994 - Dec 2001,"264,699 members"


In [53]:
import re

# Function to extract clean details
def refine_details(detail_text):
    parts = detail_text.strip().split("\n")  # Split by new lines
    
    # Extract format (e.g., "Manga")
    format_type = parts[0].split(" (")[0].strip() if len(parts) > 0 else None
    
    # Extract volume info (e.g., "24 vols")
    volume_info = re.search(r"\((.*?)\)", parts[0])
    volume_count = volume_info.group(1) if volume_info else None
    
    # Extract publication dates
    if len(parts) > 1:
        pub_dates = parts[1].split(" - ")
        start_date = pub_dates[0].strip() if len(pub_dates) > 0 else None
        end_date = pub_dates[1].strip() if len(pub_dates) > 1 else None
    else:
        start_date, end_date = None, None
    
    # Extract member count (from the last part)
    member_count = parts[-1].replace(" members", "").replace(",", "").strip() if len(parts) > 2 else None
    
    return pd.Series([format_type, volume_count, start_date, end_date, member_count])

# Apply extraction function
df_cleaned[['Format', 'Volume Count', 'Start Date', 'End Date', 'Member Count']] = df['details'].apply(refine_details)

# Drop original details column
#df_cleaned = df_cleaned.drop(columns='details',axis=1)

# Display cleaned dataset

df_cleaned.drop(columns=['Volume Info','Publication Dates','rank'],axis=1,inplace=True)


In [ ]:
df_cleaned.insert(0,'Rank',range(1,651))

In [55]:
df_cleaned

,name,rating,Format,Volume Count,Start Date,End Date,Member Count
0,Berserk,9.47,Manga,? vols,Aug 1989,,741230
1,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287722
2,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417353
3,One Piece,9.22,Manga,? vols,Jul 1997,,654245
4,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264699
...,...,...,...,...,...,...,...
645,Goukon ni Ittara Onna ga Inakatta Hanashi,8.07,Manga,? vols,Feb 2021,,5576
646,Hitoribocchi no ○○ Seikatsu,8.07,Manga,8 vols,Sep 2013,Apr 2021,25442
647,Nogi Wakaba wa Yuusha de Aru,8.07,Light Novel,2 vols,Jul 2015,Jan 2017,4899
648,Orange,8.07,Light Novel,3 vols,Jul 2015,Mar 2016,9831


In [57]:
df_cleaned['End Date']=df_cleaned['End Date'].replace('',"unfinished")

In [59]:
df_cleaned['Volume Count']=df_cleaned['Volume Count'].replace('? vols',"unfinished")

In [61]:
df_cleaned

,name,rating,Format,Volume Count,Start Date,End Date,Member Count
0,Berserk,9.47,Manga,unfinished,Aug 1989,unfinished,741230
1,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287722
2,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417353
3,One Piece,9.22,Manga,unfinished,Jul 1997,unfinished,654245
4,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264699
...,...,...,...,...,...,...,...
645,Goukon ni Ittara Onna ga Inakatta Hanashi,8.07,Manga,unfinished,Feb 2021,unfinished,5576
646,Hitoribocchi no ○○ Seikatsu,8.07,Manga,8 vols,Sep 2013,Apr 2021,25442
647,Nogi Wakaba wa Yuusha de Aru,8.07,Light Novel,2 vols,Jul 2015,Jan 2017,4899
648,Orange,8.07,Light Novel,3 vols,Jul 2015,Mar 2016,9831


In [63]:
top_5_rated = df_cleaned[['name', 'rating']].sort_values(by='rating', ascending=False).head(35)
top_5_rated

,name,rating
0,Berserk,9.47
1,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32
2,Vagabond,9.26
3,One Piece,9.22
4,Monster,9.16
5,Slam Dunk,9.08
6,Vinland Saga,9.08
7,Fullmetal Alchemist,9.04
10,Grand Blue,9.03
8,Tian Guan Cifu,9.03


In [246]:
df_cleaned.to_csv(r'C:\Users\dtrri\bot\temp2.csv')

In [51]:
for i in soup.find_all("div",class_="hoverinfo left top"):
    print(i.text)

In [15]:
soup.find_all("span",id_="info2",class_="dark_text")

[]

In [53]:
soup.find_all("div",class_="hoverinfo-contaniner")

[]

In [ ]:
<div class="detail">
              <div id="area2">
  <div id="info2" rel="m2" class="hoverinfo left top" style="left: 238.875px; top: 154.6px; display: none;"><div class="hoverinfo-contaniner"><span></span><div>
  <a href=" https://myanimelist.net/manga/2/Berserk" class="hovertitle">
    Berserk
        (1989)  </a>
</div>
<div style="margin-top: 8px; margin-bottom: 10px;">
  Guts, a former mercenary now known as the Black Swordsman, is out for revenge. After a tumultuous childhood, he finally finds someone he respects and believes he can trust, only to have everything fal
      ...
    <a href="https://myanimelist.net/manga/2/Berserk">read more</a>
  </div>

            <span class="dark_text">Genres:</span>
              Action,       Adventure,       Award Winning,       Drama,       Fantasy,       Horror<br>
              <span class="dark_text">Themes:</span>
              Gore,       Military,       Psychological<br>
              <span class="dark_text">Demographic:</span>
              Seinen<br>
  
<span class="dark_text">Status:</span>
Publishing<br>

<span class="dark_text">Volumes:</span>
Unknown<br>

<span class="dark_text">Score:</span>
9.47
<small>(scored by 372,037 users)</small><br>

<span class="dark_text">Ranked:</span>
#1<br>

<span class="dark_text">Popularity:</span>
#1<br>

<span class="dark_text">Members:</span>
741,049<br>
</div></div>
</div>
            <h3 class="manga_h3"><a class="hoverinfo_trigger fs14 fw-b" href="https://myanimelist.net/manga/2/Berserk" id="#area2" rel="#info2" style="position: relative; display: inline-block;">Berserk</a></h3>
      <div class="information di-ib mt4">
        Manga (? vols)<br>
        Aug 1989 - <br>
        741,047 members
      </div>
    </div>

In [75]:
for i in soup.find_all("span",class_="dark_text"):
    print(i.text)

Synonyms:
Japanese:
English:
Type:
Volumes:
Chapters:
Status:
Published:
Genres:
Themes:
Demographic:
Serialization:
Authors:
Score:
Ranked:
Popularity:
Members:
Favorites:


In [77]:
soup.find_all("div",class_="hoverinfo-contaniner")

[]

In [85]:
soup.find_all("div",class_="detail")

[<div class="detail">
 <div id="area2">
 <div class="hoverinfo" id="info2" rel="m2"></div>
 </div>
 <h3 class="manga_h3"><a class="hoverinfo_trigger fs14 fw-b" href="https://myanimelist.net/manga/2/Berserk" id="#area2" rel="#info2">Berserk</a></h3>
 <div class="information di-ib mt4">
         Manga (? vols)<br/>
         Aug 1989 - <br/>
         741,055 members
       </div>
 </div>,
 <div class="detail">
 <div id="area1706">
 <div class="hoverinfo" id="info1706" rel="m1706"></div>
 </div>
 <h3 class="manga_h3"><a class="hoverinfo_trigger fs14 fw-b" href="https://myanimelist.net/manga/1706/JoJo_no_Kimyou_na_Bouken_Part_7__Steel_Ball_Run" id="#area1706" rel="#info1706">JoJo no Kimyou na Bouken Part 7: Steel Ball Run</a></h3>
 <div class="information di-ib mt4">
         Manga (24 vols)<br/>
         Jan 2004 - Apr 2011<br/>
         287,647 members
       </div>
 </div>,
 <div class="detail">
 <div id="area656">
 <div class="hoverinfo" id="info656" rel="m656"></div>
 </div>
 <h3 class

In [192]:
list_=[]
for i in soup.find_all("div",class_="hoverinfo"):
    list_.append(i)

In [43]:
list_=list(list_)

In [198]:
list_

[<div class="hoverinfo" id="info2" rel="m2"></div>,
 <div class="hoverinfo" id="info1706" rel="m1706"></div>,
 <div class="hoverinfo" id="info656" rel="m656"></div>,
 <div class="hoverinfo" id="info13" rel="m13"></div>,
 <div class="hoverinfo" id="info1" rel="m1"></div>,
 <div class="hoverinfo" id="info51" rel="m51"></div>,
 <div class="hoverinfo" id="info642" rel="m642"></div>,
 <div class="hoverinfo" id="info25" rel="m25"></div>,
 <div class="hoverinfo" id="info130826" rel="m130826"></div>,
 <div class="hoverinfo" id="info162032" rel="m162032"></div>,
 <div class="hoverinfo" id="info70345" rel="m70345"></div>,
 <div class="hoverinfo" id="info143441" rel="m143441"></div>,
 <div class="hoverinfo" id="info16765" rel="m16765"></div>,
 <div class="hoverinfo" id="info4632" rel="m4632"></div>,
 <div class="hoverinfo" id="info44489" rel="m44489"></div>,
 <div class="hoverinfo" id="info657" rel="m657"></div>,
 <div class="hoverinfo" id="info3" rel="m3"></div>,
 <div class="hoverinfo" id="info

In [51]:
import regex as re
def mofo(text):
    numbers = re.findall(r'\d+', text)
    return numbers

In [200]:
dummy_list=[]
for i in list_:
    dummy_list.append(int(mofo(str(i))[0]))

In [196]:
for i in [50,100,150,200,250,300,350,400,450,500,550,600]:
    url1 = f"https://myanimelist.net/topmanga.php?limit={i}"
    page=requests.get(url1)
    soup=BeautifulSoup(page.text)
    for i in soup.find_all("div",class_="hoverinfo"):
        list_.append(i)

In [148]:
dummy_list=[]
url1 = f"https://myanimelist.net/topmanga.php?limit=100"
page=requests.get(url)
soup=BeautifulSoup(page.text)
list_=[]
for j in list_:
        dummy_list.append(int(mofo(str(j))[0]))

In [202]:
dummy_list

[2,
 1706,
 656,
 13,
 1,
 51,
 642,
 25,
 130826,
 162032,
 70345,
 143441,
 16765,
 4632,
 44489,
 657,
 3,
 1303,
 104,
 14893,
 121405,
 23751,
 34053,
 90125,
 126479,
 336,
 1224,
 35243,
 140765,
 56805,
 44227,
 104039,
 126287,
 21525,
 70261,
 9115,
 28,
 651,
 7375,
 74697,
 144267,
 147272,
 89357,
 123992,
 86119,
 100448,
 85781,
 86769,
 26,
 14483,
 7,
 102875,
 40171,
 91941,
 132678,
 44347,
 72467,
 147769,
 45143,
 116778,
 125036,
 418,
 21,
 145,
 55215,
 1133,
 1859,
 664,
 121496,
 100035,
 4,
 9726,
 117077,
 123006,
 129621,
 88660,
 98930,
 107562,
 145863,
 23900,
 26027,
 24875,
 78523,
 85968,
 103237,
 128365,
 137200,
 44,
 904,
 6978,
 81,
 745,
 81669,
 60783,
 70399,
 35513,
 132214,
 102256,
 107931,
 60553,
 756,
 9982,
 118289,
 114043,
 116312,
 145539,
 37707,
 61579,
 399,
 698,
 3006,
 13245,
 3031,
 11514,
 92559,
 87844,
 147448,
 138533,
 610,
 1325,
 2062,
 11054,
 134744,
 23390,
 66517,
 105084,
 164926,
 22,
 8157,
 11327,
 102,
 3083,
 

In [366]:
dummy_list[411:441]

[14531,
 16593,
 348,
 585,
 731,
 1075,
 1208,
 1320,
 3731,
 5651,
 17552,
 131008,
 132962,
 133352,
 134997,
 58027,
 103063,
 106204,
 107548,
 108495,
 108566,
 102330,
 18074,
 24705,
 26736,
 37523,
 510,
 7149,
 15805,
 70801]

In [429]:
genre=[]
for i in dummy_list[621:651]:
    url = f"https://myanimelist.net/manga/{i}"
    page=requests.get(url)
    soup=BeautifulSoup(page.text)
    a=[]
    for j in soup.find_all("span",itemprop="genre"):
        a.append(j.text)
    genre.append(a)

In [431]:
genre

[['Award Winning', 'Drama', 'Mystery', 'Supernatural', 'Detective', 'Shounen'],
 ['Comedy', 'Romance', 'School', 'Shoujo'],
 ['Action', 'Comedy', 'Drama', 'Ecchi', 'Martial Arts', 'Shounen'],
 ['Drama', 'Fantasy', 'Sci-Fi', 'Slice of Life', 'Supernatural', 'Seinen'],
 ['Drama', 'Adult Cast', 'Psychological', 'Strategy Game', 'Seinen'],
 ['Award Winning', 'Drama', 'Historical', 'Seinen'],
 ['Boys Love', 'Comedy', 'Adult Cast', 'Organized Crime'],
 ['Comedy', 'Gag Humor', 'Seinen'],
 ['Action', 'Supernatural', 'Historical', 'Shoujo'],
 ['Fantasy'],
 ['Boys Love', 'Drama', 'Supernatural'],
 ['Boys Love', 'Comedy', 'Supernatural', 'Adult Cast', 'Workplace'],
 ['Adventure', 'Fantasy', 'Slice of Life', 'Psychological'],
 ['Action', 'Supernatural', 'Shounen'],
 ['Drama', 'Fantasy', 'Romance'],
 ['Action', 'Adventure', 'Fantasy', 'Super Power', 'Shounen'],
 ['Drama', 'Romance', 'Supernatural'],
 ['Action', 'Adventure', 'Fantasy', 'Martial Arts', 'Shounen'],
 ['Comedy', 'Romance', 'School', 'Sh

In [102]:
import pandas as pd
df=pd.read_csv(r'C:\Users\dtrri\bot\temp2.csv')

In [110]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [433]:
df['Genre'][621:651]=genre

C:\Users\dtrri\AppData\Local\Temp\ipykernel_26516\2486546725.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Genre'][621:651]=genre


In [435]:
df

,Rank,name,rating,Format,Volume Count,Start Date,End Date,Member Count,Genre
0,1,Berserk,9.47,Manga,unfinished,Aug 1989,unfinished,740803,"[Action, Adventure, Award Winning, Drama, Fant..."
1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287531,"[Action, Adventure, Mystery, Supernatural, His..."
2,3,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417041,"[Action, Adventure, Award Winning, Historical,..."
3,4,One Piece,9.22,Manga,unfinished,Jul 1997,unfinished,653920,"[Action, Adventure, Fantasy, Shounen]"
4,5,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264525,"[Award Winning, Drama, Mystery, Adult Cast, Ps..."
...,...,...,...,...,...,...,...,...,...
645,646,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...,8.77,Light Novel,15 vols,Jan 2020,Nov 2024,39292,"[Action, Fantasy, Super Power, Urban Fantasy, ..."
646,647,"Jumyou wo Kaitotte Moratta. Ichinen ni Tsuki, ...",8.77,Manga,3 vols,Aug 2016,Oct 2017,175768,"[Drama, Fantasy, Horror, Mahou Shoujo, School]"
647,648,Dungeon Meshi,8.76,Manga,14 vols,Feb 2014,Sep 2023,93656,"[Drama, Mystery, Sci-Fi]"
648,649,Hunter x Hunter,8.76,Manga,unfinished,Mar 1998,unfinished,290839,"[Romance, Shoujo]"


In [437]:
df["Genre"].value_counts()

Genre
[Drama]                                                        5
[Award Winning, Sports, School, Team Sports, Shounen]          5
[Comedy, Romance, School, Shoujo]                              5
[Drama, Horror, Mystery, Psychological]                        4
[Action, Adventure, Fantasy, Shounen]                          4
                                                              ..
[Drama, Josei]                                                 1
[Fantasy, Mystery, Seinen]                                     1
[Action, Adventure, Comedy, Fantasy, Isekai, Reincarnation]    1
[Drama, Romance, Historical, Military, Shoujo]                 1
[Comedy, Romance, Adult Cast, Crossdressing]                   1
Name: count, Length: 546, dtype: int64

In [439]:
df.to_csv(r'C:\Users\dtrri\bot\EDA_PROJECT.csv')

In [33]:
df1=pd.read_csv(r'C:\Users\dtrri\bot\EDA_PROJECT.csv')

In [35]:
df1

,Unnamed: 0,Rank,name,rating,Format,Volume Count,Start Date,End Date,Member Count,Genre
0,0,1,Berserk,9.47,Manga,unfinished,Aug 1989,unfinished,740803,"['Action', 'Adventure', 'Award Winning', 'Dram..."
1,1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287531,"['Action', 'Adventure', 'Mystery', 'Supernatur..."
2,2,3,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417041,"['Action', 'Adventure', 'Award Winning', 'Hist..."
3,3,4,One Piece,9.22,Manga,unfinished,Jul 1997,unfinished,653920,"['Action', 'Adventure', 'Fantasy', 'Shounen']"
4,4,5,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264525,"['Award Winning', 'Drama', 'Mystery', 'Adult C..."
...,...,...,...,...,...,...,...,...,...,...
645,645,646,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...,8.77,Light Novel,15 vols,Jan 2020,Nov 2024,39292,"['Action', 'Fantasy', 'Super Power', 'Urban Fa..."
646,646,647,"Jumyou wo Kaitotte Moratta. Ichinen ni Tsuki, ...",8.77,Manga,3 vols,Aug 2016,Oct 2017,175768,"['Drama', 'Fantasy', 'Horror', 'Mahou Shoujo',..."
647,647,648,Dungeon Meshi,8.76,Manga,14 vols,Feb 2014,Sep 2023,93656,"['Drama', 'Mystery', 'Sci-Fi']"
648,648,649,Hunter x Hunter,8.76,Manga,unfinished,Mar 1998,unfinished,290839,"['Romance', 'Shoujo']"


In [ ]:
"rank":rank,
                 'name':name,
                "details":details,
                "rating":rating

In [71]:
df1.columns

Index(['Unnamed: 0', 'Rank', 'name', 'rating', 'Format', 'Volume Count',
       'Start Date', 'End Date', 'Member Count', 'Genre'],
      dtype='object')

In [73]:
df_cleaned.columns

Index(['name', 'rating', 'Format', 'Volume Count', 'Start Date', 'End Date',
       'Member Count'],
      dtype='object')

In [65]:
df1['name']=df_cleaned['name']

In [67]:
df1['rating']=df_cleaned['rating']

In [75]:
df1['Format']=df_cleaned['Format']

In [77]:
df1['Volume Count']=df_cleaned['Volume Count']

In [79]:
df1['Start Date']=df_cleaned['Start Date']

In [83]:
df1['End Date']=df_cleaned['End Date']

In [81]:
df1['Member Count']=df_cleaned['Member Count']

In [85]:
df1

,Unnamed: 0,Rank,name,rating,Format,Volume Count,Start Date,End Date,Member Count,Genre
0,0,1,Berserk,9.47,Manga,unfinished,Aug 1989,unfinished,741230,"['Action', 'Adventure', 'Award Winning', 'Dram..."
1,1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287722,"['Action', 'Adventure', 'Mystery', 'Supernatur..."
2,2,3,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417353,"['Action', 'Adventure', 'Award Winning', 'Hist..."
3,3,4,One Piece,9.22,Manga,unfinished,Jul 1997,unfinished,654245,"['Action', 'Adventure', 'Fantasy', 'Shounen']"
4,4,5,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264699,"['Award Winning', 'Drama', 'Mystery', 'Adult C..."
...,...,...,...,...,...,...,...,...,...,...
645,645,646,Goukon ni Ittara Onna ga Inakatta Hanashi,8.07,Manga,unfinished,Feb 2021,unfinished,5576,"['Action', 'Fantasy', 'Super Power', 'Urban Fa..."
646,646,647,Hitoribocchi no ○○ Seikatsu,8.07,Manga,8 vols,Sep 2013,Apr 2021,25442,"['Drama', 'Fantasy', 'Horror', 'Mahou Shoujo',..."
647,647,648,Nogi Wakaba wa Yuusha de Aru,8.07,Light Novel,2 vols,Jul 2015,Jan 2017,4899,"['Drama', 'Mystery', 'Sci-Fi']"
648,648,649,Orange,8.07,Light Novel,3 vols,Jul 2015,Mar 2016,9831,"['Romance', 'Shoujo']"


In [87]:
top_5_rated = df1[['name', 'rating']].sort_values(by='rating', ascending=False).head(35)
top_5_rated

,name,rating
0,Berserk,9.47
1,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32
2,Vagabond,9.26
3,One Piece,9.22
4,Monster,9.16
5,Slam Dunk,9.08
6,Vinland Saga,9.08
7,Fullmetal Alchemist,9.04
10,Grand Blue,9.03
8,Tian Guan Cifu,9.03


In [89]:
unique_titles_count = df1['name'].nunique()
unique_titles_count

629

In [91]:
df1['name'].value_counts()

name
Orange                                                 2
Mo Dao Zu Shi                                          2
Bungou Stray Dogs: Dazai, Chuuya, Juugosai             2
Kimi no Suizou wo Tabetai                              2
Roxana                                                 2
                                                      ..
Tomodachi Game                                         1
Shousetsu Akatsuki no Yona: Onaji Tsuki no Shita de    1
Gunnm                                                  1
Kaichou wa Maid-sama!                                  1
Mahou Shoujo Madoka★Magica                             1
Name: count, Length: 629, dtype: int64

In [115]:
df1[['name', 'rating', 'Format', 'Volume Count', 'Start Date', 'End Date','Member Count']].duplicated().sum()

2

In [109]:
df_cleaned.duplicated().sum()

2

In [103]:
df_cleaned[df_cleaned.duplicated()]

,name,rating,Format,Volume Count,Start Date,End Date,Member Count
560,Roxana,8.12,Manhwa,unfinished,Jan 2021,unfinished,13058
561,"Bungou Stray Dogs: Dazai, Chuuya, Juugosai",8.12,Manga,4 vols,Sep 2022,Aug 2024,2138


In [105]:
df_cleaned.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
645    False
646    False
647    False
648    False
649    False
Length: 650, dtype: bool

In [117]:
df1

,Unnamed: 0,Rank,name,rating,Format,Volume Count,Start Date,End Date,Member Count,Genre
0,0,1,Berserk,9.47,Manga,unfinished,Aug 1989,unfinished,741230,"['Action', 'Adventure', 'Award Winning', 'Dram..."
1,1,2,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,9.32,Manga,24 vols,Jan 2004,Apr 2011,287722,"['Action', 'Adventure', 'Mystery', 'Supernatur..."
2,2,3,Vagabond,9.26,Manga,37 vols,Sep 1998,May 2015,417353,"['Action', 'Adventure', 'Award Winning', 'Hist..."
3,3,4,One Piece,9.22,Manga,unfinished,Jul 1997,unfinished,654245,"['Action', 'Adventure', 'Fantasy', 'Shounen']"
4,4,5,Monster,9.16,Manga,18 vols,Dec 1994,Dec 2001,264699,"['Award Winning', 'Drama', 'Mystery', 'Adult C..."
...,...,...,...,...,...,...,...,...,...,...
645,645,646,Goukon ni Ittara Onna ga Inakatta Hanashi,8.07,Manga,unfinished,Feb 2021,unfinished,5576,"['Action', 'Fantasy', 'Super Power', 'Urban Fa..."
646,646,647,Hitoribocchi no ○○ Seikatsu,8.07,Manga,8 vols,Sep 2013,Apr 2021,25442,"['Drama', 'Fantasy', 'Horror', 'Mahou Shoujo',..."
647,647,648,Nogi Wakaba wa Yuusha de Aru,8.07,Light Novel,2 vols,Jul 2015,Jan 2017,4899,"['Drama', 'Mystery', 'Sci-Fi']"
648,648,649,Orange,8.07,Light Novel,3 vols,Jul 2015,Mar 2016,9831,"['Romance', 'Shoujo']"


In [119]:
df1.to_csv(r'C:\Users\dtrri\bot\EDA_PROJECT_v2.csv',index=False)